# Notebook 4: Model and Tool Analysis

**Goal:** Compare models and tools without hiding uncertainty or repository imbalance.

**Inputs:** `evaluation_attempts.csv`, `evaluation_candidates.csv`, `evaluation_repositories.csv`

**RQs addressed:** RQ5

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from analysis.statistics import bootstrap_ci, repository_blocked_bootstrap
from analysis.plots import (
    metric_delta_distribution,
    cost_vs_improvement_scatter,
    robustness_heatmap,
)

sns.set_theme(style='whitegrid')

DATA_DIR   = Path('../output')
attempts   = pd.read_csv(DATA_DIR / 'evaluation_attempts.csv')
candidates = pd.read_csv(DATA_DIR / 'evaluation_candidates.csv')
repos      = pd.read_csv(DATA_DIR / 'evaluation_repositories.csv') if (DATA_DIR / 'evaluation_repositories.csv').exists() else pd.DataFrame()

## 1. Model / Tool Participation and Data Completeness

In [ ]:
# LLM models
llm = attempts[attempts.get('lane', pd.Series()) == 'llm']
if 'model' in llm.columns:
    print('LLM models:')
    print(llm.groupby('model').size().sort_values(ascending=False))

# Agentic tools
agentic = attempts[attempts.get('lane', pd.Series()) == 'agentic']
if 'tool_id' in agentic.columns:
    print('\nAgentic tools:')
    print(agentic.groupby('tool_id').size().sort_values(ascending=False))

## 2. Outcome Rates with Confidence Intervals

In [ ]:
def ci_for_group(df, group_col, metric_col='validated_success'):
    rows = []
    for grp, sub in df.groupby(group_col):
        vals = sub[metric_col].dropna().astype(float)
        if len(vals) < 2:
            continue
        ci = bootstrap_ci(vals, stat_fn=np.mean)
        rows.append({'group': grp, 'n': len(vals), 'mean': ci.estimate,
                     'ci_lower': ci.lower, 'ci_upper': ci.upper})
    return pd.DataFrame(rows).sort_values('mean', ascending=False)

if 'model' in llm.columns and 'validated_success' in llm.columns:
    model_ci = ci_for_group(llm, 'model')
    display(model_ci)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(model_ci['group'], model_ci['mean'],
            xerr=[model_ci['mean'] - model_ci['ci_lower'],
                  model_ci['ci_upper'] - model_ci['mean']],
            capsize=4, color='steelblue', alpha=0.8)
    ax.set_xlabel('Success rate')
    ax.set_title('LLM model success rate (with 95% bootstrap CI)')
    plt.tight_layout()
    plt.show()

In [ ]:
if 'tool_id' in agentic.columns and 'validated_success' in agentic.columns:
    tool_ci = ci_for_group(agentic, 'tool_id')
    display(tool_ci)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(tool_ci['group'], tool_ci['mean'],
            xerr=[tool_ci['mean'] - tool_ci['ci_lower'],
                  tool_ci['ci_upper'] - tool_ci['mean']],
            capsize=4, color='darkorange', alpha=0.8)
    ax.set_xlabel('Success rate')
    ax.set_title('Agentic tool success rate (with 95% bootstrap CI)')
    plt.tight_layout()
    plt.show()

## 3. Metric Improvements

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if 'model' in llm.columns and 'coverage_delta' in llm.columns:
    metric_delta_distribution(llm, 'coverage_delta', 'model', ax=axes[0])
if 'tool_id' in agentic.columns and 'coverage_delta' in agentic.columns:
    metric_delta_distribution(agentic, 'coverage_delta', 'tool_id', ax=axes[1])
plt.tight_layout()
plt.show()

## 4. Cost Efficiency

In [ ]:
# Validated successes per 1k tokens (where token data available)
for label, df, group_col in [('LLM models', llm, 'model'), ('Agentic tools', agentic, 'tool_id')]:
    if group_col in df.columns and 'total_tokens' in df.columns and 'validated_success' in df.columns:
        eff = df.groupby(group_col).apply(
            lambda g: g['validated_success'].sum() / (g['total_tokens'].sum() / 1000)
            if g['total_tokens'].sum() > 0 else float('nan')
        )
        print(f'{label} — successes per 1k tokens:')
        print(eff.sort_values(ascending=False))

## 5. Robustness Across Repositories

In [ ]:
# Robustness heatmap: model/tool × repository
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

if 'model' in llm.columns and 'repository_key' in llm.columns and 'validated_success' in llm.columns:
    robustness_heatmap(llm, 'model', 'repository_key', 'validated_success', ax=axes[0])

if 'tool_id' in agentic.columns and 'repository_key' in agentic.columns and 'validated_success' in agentic.columns:
    robustness_heatmap(agentic, 'tool_id', 'repository_key', 'validated_success', ax=axes[1])

plt.tight_layout()
plt.show()

## 6. Best-by-Category Summary

In [ ]:
# Scaffold: fill in once all metric columns are confirmed
categories = {
    'best_overall':       ('validated_success', attempts),
    'best_coverage':      ('coverage_delta',    attempts),
    'best_mutation':      ('mutation_score_delta', attempts),
}

results = []
for category, (metric, df) in categories.items():
    for producer_col in ('model', 'tool_id'):
        if producer_col in df.columns and metric in df.columns:
            best = df.groupby(producer_col)[metric].mean().idxmax()
            score = df.groupby(producer_col)[metric].mean().max()
            results.append({'category': category, 'producer_col': producer_col, 'winner': best, 'score': score})

pd.DataFrame(results)